# Global Superstore: Sales & Profit Analysis

> **Beginner-Friendly | EDA + ML | Regression**

---

## 📌 What This Notebook Covers

In this notebook, we will:
1. **Load & Explore** the Global Superstore dataset (51,290 orders)
2. **Clean** the data (handle types, drop irrelevant columns)
3. **Visualize** key patterns using charts
4. **Build ML Models** to predict `Profit` from order features
5. **Compare Models** and pick the best one
6. **Conclude** with business insights

---

### 🗂️ Dataset Info
- **Source:** Global Superstore Orders
- **Rows:** 51,290 | **Columns:** 27
- **Target Variable:** `Profit` (continuous → Regression task)
- **Key Features:** Sales, Discount, Quantity, Shipping Cost, Category, Segment, Market

## 📦 Step 1: Import Libraries

We import all the tools we need:
- `pandas` → data manipulation
- `numpy` → math operations
- `matplotlib` / `seaborn` → charts
- `sklearn` → ML models
- `xgboost` → powerful gradient boosting model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ Libraries imported successfully!')

## 📂 Step 2: Load the Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/global-superstore/Global_Superstore.csv', encoding='latin1')

print(f'Shape: {df.shape}')  # rows x columns
df.head()

## 🔍 Step 3: Exploratory Data Analysis (EDA)

Before building any model, we explore the data to understand it.

In [ ]:
# Basic info
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Missing Values ---')
print(df.isnull().sum())

In [ ]:
# Statistical summary of numeric columns
df[['Sales', 'Profit', 'Discount', 'Quantity', 'Shipping Cost']].describe().round(2)

### 📊 3.1 Profit Distribution

Let's see how profit is spread across all orders. Negative values mean losses!

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of Profit
axes[0].hist(df['Profit'], bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', label='Break-even (0)')
axes[0].set_title('Distribution of Profit')
axes[0].set_xlabel('Profit')
axes[0].set_ylabel('Count')
axes[0].legend()

# Profit by Category
category_profit = df.groupby('Category')['Profit'].sum().sort_values()
axes[1].barh(category_profit.index, category_profit.values, color=['#e74c3c','#2ecc71','#3498db'])
axes[1].set_title('Total Profit by Category')
axes[1].set_xlabel('Total Profit')

plt.tight_layout()
plt.show()

### 📊 3.2 Sales vs Profit (Scatter Plot)

Does higher sales always mean higher profit? Let's find out.

In [ ]:
plt.figure(figsize=(10, 5))
# Cap extreme values for better visibility
sample = df[(df['Sales'] < 5000) & (df['Profit'].between(-1000, 1000))]

plt.scatter(sample['Sales'], sample['Profit'],
            c=sample['Discount'], cmap='RdYlGn_r', alpha=0.4, s=15)
plt.colorbar(label='Discount Rate')
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.title('Sales vs Profit (colored by Discount)')
plt.xlabel('Sales')
plt.ylabel('Profit')
plt.tight_layout()
plt.show()

print('💡 Insight: High discount (red dots) often leads to negative profit!')

### 📊 3.3 Profit by Market & Segment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Profit by Market
market_profit = df.groupby('Market')['Profit'].sum().sort_values(ascending=False)
market_profit.plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Total Profit by Market')
axes[0].set_xlabel('Market')
axes[0].set_ylabel('Profit')
axes[0].tick_params(axis='x', rotation=45)

# Profit by Segment
segment_profit = df.groupby('Segment')['Profit'].sum()
axes[1].pie(segment_profit, labels=segment_profit.index,
            autopct='%1.1f%%', colors=['#3498db','#2ecc71','#e74c3c'],
            startangle=140)
axes[1].set_title('Profit Share by Segment')

plt.tight_layout()
plt.show()

### 📊 3.4 Correlation Heatmap

Which numeric features are most related to `Profit`?

In [ ]:
numeric_cols = ['Sales', 'Profit', 'Discount', 'Quantity', 'Shipping Cost']
corr = df[numeric_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

print('💡 Discount has a negative correlation with Profit — more discount = less profit!')

## 🛠️ Step 4: Data Preprocessing

We prepare the data so ML models can understand it:
- Drop columns not useful for prediction
- Encode categorical (text) columns as numbers

In [ ]:
# Drop non-predictive or leaky columns
drop_cols = ['Row ID', 'Order ID', 'Customer ID', 'Customer Name',
             'Product ID', 'Product Name', 'Order Date', 'Ship Date',
             'ji_lu-shu', 'Market2']  # duplicate/irrelevant

df_model = df.drop(columns=drop_cols)

# Encode categorical columns
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print('Categorical columns to encode:', cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

print('\n✅ Encoding done! Shape:', df_model.shape)
df_model.head(3)

In [ ]:
# Define features (X) and target (y)
X = df_model.drop(columns=['Profit'])
y = df_model['Profit']

# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} rows')
print(f'Testing set : {X_test.shape[0]} rows')

## 🤖 Step 5: Train Multiple ML Models

We try 5 models and compare which one predicts profit best.

| Model | Type |
|---|---|
| Linear Regression | Baseline |
| Decision Tree | Simple tree |
| Random Forest | Many trees averaged |
| Gradient Boosting | Trees built sequentially |
| XGBoost | Optimized boosting |

In [ ]:
# Define all models
models = {
    'Linear Regression':    LinearRegression(),
    'Decision Tree':        DecisionTreeRegressor(random_state=42),
    'Random Forest':        RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost':              XGBRegressor(n_estimators=100, random_state=42,
                                         learning_rate=0.1, verbosity=0)
}

# Train and evaluate each model
results = []

for name, model in models.items():
    model.fit(X_train, y_train)          # Train
    y_pred = model.predict(X_test)       # Predict

    r2  = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({'Model': name, 'R² Score': round(r2, 4),
                    'MAE': round(mae, 2), 'RMSE': round(rmse, 2)})
    print(f'{name:22s} | R²={r2:.4f} | MAE={mae:.2f} | RMSE={rmse:.2f}')

results_df = pd.DataFrame(results).sort_values('R² Score', ascending=False)
print('\n✅ Training complete!')

## 📊 Step 6: Model Comparison

In [ ]:
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['R² Score', 'MAE', 'RMSE']
colors  = ['#2ecc71', '#e74c3c', '#e67e22']

for ax, metric, color in zip(axes, metrics, colors):
    sorted_df = results_df.sort_values(metric, ascending=(metric != 'R² Score'))
    ax.barh(sorted_df['Model'], sorted_df[metric], color=color, edgecolor='white')
    ax.set_title(f'Model Comparison — {metric}')
    ax.set_xlabel(metric)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🏆 Step 7: Best Model — Deep Dive

We use the best model to look at:
- **Actual vs Predicted** scatter plot
- **Feature Importance** — which features drive profit the most

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]
y_pred_best = best_model.predict(X_test)

print(f'🏆 Best Model: {best_name}')
print(f'   R² Score : {r2_score(y_test, y_pred_best):.4f}')

# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cap for readability
mask = (y_test > -500) & (y_test < 1000)
axes[0].scatter(y_test[mask], y_pred_best[mask], alpha=0.3, s=10, color='steelblue')
axes[0].plot([-500, 1000], [-500, 1000], 'r--', lw=1.5, label='Perfect Prediction')
axes[0].set_title(f'{best_name} — Actual vs Predicted')
axes[0].set_xlabel('Actual Profit')
axes[0].set_ylabel('Predicted Profit')
axes[0].legend()

# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=X.columns)
    importance.nlargest(12).sort_values().plot(kind='barh', ax=axes[1],
                                               color='darkorange', edgecolor='white')
    axes[1].set_title('Top 12 Feature Importances')
    axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

## ✅ Step 8: Conclusion

---

### 🔑 Key Findings from EDA

| Finding | Detail |
|---|---|
| 💸 Discount kills profit | Orders with high discount (>0.4) frequently result in losses |
| 🏆 Best market | Asia Pacific & Europe generate highest profit |
| 📦 Best category | Technology leads, Furniture lags |
| 🧑‍💼 Best segment | Consumer segment contributes the most profit share |

---

### 🤖 Model Performance Summary

- **Linear Regression** performs poorly — profit is not linearly related to features
- **Random Forest & XGBoost** achieve the best R² scores, showing tree-based models handle this tabular data well
- **Top predictors of profit**: `Sales`, `Discount`, `Shipping Cost`, `Quantity`

---

### 💼 Business Recommendations

1. **Cap discounts** — avoid offering >30% discount; it consistently results in losses
2. **Focus on Technology** — highest profit margin category globally
3. **Optimize shipping** — high shipping cost reduces net profit on small orders
4. **Double down on Asia Pacific** — the most profitable market overall

---

*If you found this notebook helpful, please consider upvoting! ⬆️*